# Cross branches vs. chain reweighting

PROfit combines spline nuisance parameters *multiplicatively*, so the z-expansion response it
uses is $\prod_i s_i(\eta_i)$. The true per-bin response is an exact multivariate quadratic in
$\eta$ *with* cross terms $\eta_i\eta_j$, which that product cannot represent (see
`../../notes/spline_factorization.md`). Two independent corrections exist, and this notebook
checks that they agree:

| | response | source |
|---|---|---|
| **A** | $\prod_i s_i(\eta_i)$ | `xml_testing/nocross/` fits |
| **B** | exact | A's chain importance-reweighted by $w=e^{+\Delta\chi^2_{\rm data}/2}$ |
| **C** | exact | `xml_testing/cross/` fits, `type="spline_cross_quad"` |

**B $\approx$ C is the closure test.** They reach the same target by completely different routes:
B corrects a wrong fit after the fact, C fits the right model. Note that the product form is
wrong by *more* than the cross term -- it also generates spurious $\eta_i^2\eta_j^2$ and
$\eta_i\eta_j^2$ pieces -- whereas the `spline_cross_quad` factor
$1+\sum_i(s_i-1)+\sum_{i<j}e_{ij}\eta_i\eta_j$ is additive and exact for a degree-$\le2$
response. So C is the reference, not merely a third approximation.

Two quantities give the comparison its scale:

* **effect size** $|{\rm A}-{\rm C}|$ -- how much the factorization error actually moves the
  posterior. If this is at the noise floor, the fit is insensitive and the test has no power
  there (expected for the LQCD-constrained priors).
* **noise floor**, the batch-means standard error of each chain's posterior mean. Two
  independent chains' means differ by this much from sampling alone, so without it "B and C
  agree to $0.01\sigma$" means nothing. Batch means account for MCMC autocorrelation, which
  a naive $\sigma/\sqrt{N}$ does not, and they make a second-seed run unnecessary.

A fit **passes** when $|{\rm B}-{\rm C}|$ is within 3 of those standard errors. It is
**informative** when the effect size is, i.e. when the factorization error was measurable
at all.

Prerequisites: `xml/run_all.sh` (with `--with-covar`, which is what writes the `Covariance`
directory notebook 12 reads), then notebook 12 for the $\Delta\chi^2$ grids, then
`xml_testing/run_test.sh` for A and C.


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Locate ma_zexp/python/scripts whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
helper_dir = next(
    (parent / 'ma_zexp' / 'python' / 'scripts' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if helper_dir is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FIGURE_ROOT, PUBLICATION_RC, SPECS, load_fit, prior_label,
)
from spline_reweighting import (
    compute_importance_weights, describe_ess, load_dchi2_grid, uniform_weights,
)

mpl.rcParams.update(PUBLICATION_RC)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 200)

REPO_MA_ZEXP = helper_dir.parents[1]
TABLE_DIR = REPO_MA_ZEXP / 'tables' / 'cross_validation'
FIG_DIR = FIGURE_ROOT / 'cross_validation'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Variant trees written by xml_testing/run_test.sh. Below each root the layout is the
# usual <suite>/<fit>/, which is why load_fit only needs its data_root overridden.
CROSSTEST_ROOT = Path('/nevis/hopper/data/epelaez/axial_mass_crosstest')
NOCROSS_DIR = CROSSTEST_ROOT / 'nocross'
CROSS_DIR = CROSSTEST_ROOT / 'cross'
NOCROSS_SEED2_DIR = CROSSTEST_ROOT / 'nocross_seed2'

SUITES = ('nuwro_fit_results', 'opendata_fit_results')
SUITE_LABEL = {'nuwro_fit_results': 'NuWro fake data', 'opendata_fit_results': 'Open data'}
FITS = ('minerva_k6', 'minerva_k7', 'minerva_k6_uniform', 'lqcd_k6')

SPEC_BY_KEY = {spec.key: spec for spec in SPECS}
missing = [f for f in FITS if f not in SPEC_BY_KEY]
assert not missing, f'unknown fits: {missing}'

BURN_IN, THIN = 0, 1

print('cross-test root:', CROSSTEST_ROOT)
for name, d in (('nocross', NOCROSS_DIR), ('cross', CROSS_DIR), ('nocross_seed2', NOCROSS_SEED2_DIR)):
    print(f'  {name:<14} {"present" if d.is_dir() else "MISSING"}  {d}')


## Section 1: what has been run

`load_fit` returns `None` when a fit directory has no `*_v1_PROfile.root`, so the notebook
reports what is available rather than failing on the first gap. The seed-2 variant is optional:
without it there is no noise floor and the verdicts fall back to a fixed threshold.


In [ ]:
def available(root, suite, fit):
    directory = root / suite / fit
    return bool(sorted(directory.glob('*_v1_PROfile.root')))


rows = []
for suite in SUITES:
    for fit in FITS:
        rows.append(dict(
            suite=SUITE_LABEL[suite], fit=fit,
            A=available(NOCROSS_DIR, suite, fit),
            C=available(CROSS_DIR, suite, fit),
            A_seed2=available(NOCROSS_SEED2_DIR, suite, fit),
        ))
INVENTORY = pd.DataFrame(rows)
display(INVENTORY)

ready = INVENTORY[INVENTORY.A & INVENTORY.C]
if ready.empty:
    raise RuntimeError('No (A, C) pair is available yet -- run xml_testing/run_test.sh first.')
HAVE_NOISE_FLOOR = bool(INVENTORY.A_seed2.all())
if not HAVE_NOISE_FLOOR:
    warnings.warn('No seed-2 run found; verdicts will use a fixed 0.05 sigma threshold '
                  'instead of the measured MCMC noise floor.')
print(f'{len(ready)} of {len(INVENTORY)} (suite, fit) combinations have both A and C.')


## Section 2: load the three variants

A and B come from the **same** chain: it is loaded once and then weighted two ways, uniformly
for A and with the notebook-12 importance weights for B. Sharing the chain makes A and B exactly
paired, so any A$-$B difference is the reweighting alone and carries no MCMC noise.

The weights are computed here rather than in `postfit_physical_parameters`: the production XMLs
now carry `spline_cross_quad`, so that module applies no correction at all and this notebook is
the only place the reweighting route still exists.

C is a separate fit, so B$-$C *does* carry the sampling noise of two independent chains -- which
is exactly what the noise floor calibrates.

A missing $\Delta\chi^2$ grid raises `FileNotFoundError`; run notebook 12 against the new
production first.


In [ ]:
def load_variant(fit, suite, data_root, reweight):
    '''Load one variant's chain; with reweight=True also attach the B weights.'''
    result = load_fit(SPEC_BY_KEY[fit], suite, burn_in=BURN_IN, thin=THIN,
                      data_root=data_root)
    if result is None:
        raise FileNotFoundError(f'{fit}/{suite}: no PROfile file under {data_root}')
    if not reweight:
        result['weights'] = uniform_weights(len(result['samples']))
        result['ess'] = float(len(result['samples']))
        return result
    try:
        grid = load_dchi2_grid(fit, suite)
    except FileNotFoundError as exc:
        raise FileNotFoundError(
            f'{fit}/{suite}: no dchi2 grid for the reweighting ({exc}). Run '
            f'12_spline_factorization_validation.ipynb against the current production first.'
        ) from exc
    _, weights, ess, _ = compute_importance_weights(
        result['eta_samples'], grid, verbose=False
    )
    result['weights'], result['ess'] = weights, ess
    return result


LOADED = {}
for _, row in ready.iterrows():
    suite = next(s for s in SUITES if SUITE_LABEL[s] == row.suite)
    fit = row.fit
    # one chain, two weightings: A (uniform) and B (reweighted)
    ab = load_variant(fit, suite, NOCROSS_DIR, True)
    c = load_variant(fit, suite, CROSS_DIR, False)
    a2 = load_variant(fit, suite, NOCROSS_SEED2_DIR, False) if row.A_seed2 else None
    LOADED[(fit, suite)] = dict(ab=ab, c=c, a2=a2)
    n = len(ab['eta_samples'])
    print(f"{fit:<20} {SUITE_LABEL[suite]:<16} chain {n:>7,} samples   "
          f"{describe_ess(ab['ess'], n)}")


## Section 3: metrics

Everything is computed in $\eta$ space (the fit coordinates, where the posterior is
approximately standardized) and again in the physical $z$-expansion coefficients $a_i$.

* mean shift, quoted in units of the A posterior width $\sigma_{\rm A}$ so the numbers are
  comparable across parameters and fits;
* width ratio $\sigma/\sigma_{\rm A}$;
* the largest difference between the weighted empirical CDFs of the two 1D marginals (a
  weighted Kolmogorov-Smirnov distance), which unlike the mean is sensitive to a change of
  shape rather than only of location.


In [ ]:
def wmean(x, w):
    return np.average(x, weights=w, axis=0)


def batch_se(x, w, n_batches=32):
    '''Batch-means standard error of the weighted mean, per column.

    Splitting the chain into blocks and taking the scatter of the block means
    accounts for MCMC autocorrelation, which a naive sigma/sqrt(N) ignores. This
    is the noise floor: it says how far two independent chains' means may sit
    apart by sampling alone, so it replaces the second-seed run the earlier
    version of this notebook needed.
    '''
    n = len(x) // n_batches * n_batches
    xb = x[:n].reshape(n_batches, -1, x.shape[1])
    wb = w[:n].reshape(n_batches, -1)
    means = np.einsum('bij,bi->bj', xb, wb) / wb.sum(axis=1)[:, None]
    return means.std(axis=0, ddof=1) / np.sqrt(n_batches)


def wstd(x, w):
    mu = wmean(x, w)
    return np.sqrt(np.average((x - mu) ** 2, weights=w, axis=0))


def weighted_ks(x1, w1, x2, w2):
    '''Largest gap between two weighted empirical CDFs of a 1D sample.'''
    grid = np.union1d(x1, x2)
    o1, o2 = np.argsort(x1), np.argsort(x2)
    c1 = np.cumsum(w1[o1]) / np.sum(w1)
    c2 = np.cumsum(w2[o2]) / np.sum(w2)
    f1 = np.interp(grid, x1[o1], c1, left=0.0, right=1.0)
    f2 = np.interp(grid, x2[o2], c2, left=0.0, right=1.0)
    return float(np.max(np.abs(f1 - f2)))


def compare(space, names, xa, wa, xb, wb, xc, wc, xa2, wa2):
    '''Per-parameter A/B/C comparison, normalized to the A posterior width.'''
    sigma_a = wstd(xa, wa)
    out = []
    for k, name in enumerate(names):
        row = dict(space=space, parameter=name, sigma_A=sigma_a[k])
        mu = {'A': wmean(xa, wa)[k], 'B': wmean(xb, wb)[k], 'C': wmean(xc, wc)[k]}
        sd = {'A': sigma_a[k], 'B': wstd(xb, wb)[k], 'C': wstd(xc, wc)[k]}
        se = {'A': batch_se(xa, wa), 'B': batch_se(xb, wb), 'C': batch_se(xc, wc)}
        row['mean_shift_A_C'] = (mu['A'] - mu['C']) / sigma_a[k]
        row['mean_shift_B_C'] = (mu['B'] - mu['C']) / sigma_a[k]
        # noise on each difference: two independent chains, so the SEs add in quadrature
        row['se_A_C'] = np.hypot(se['A'][k], se['C'][k]) / sigma_a[k]
        row['se_B_C'] = np.hypot(se['B'][k], se['C'][k]) / sigma_a[k]
        row['z_A_C'] = row['mean_shift_A_C'] / row['se_A_C']
        row['z_B_C'] = row['mean_shift_B_C'] / row['se_B_C']
        row['width_ratio_B_C'] = sd['B'] / sd['C']
        row['ks_A_C'] = weighted_ks(xa[:, k], wa, xc[:, k], wc)
        row['ks_B_C'] = weighted_ks(xb[:, k], wb, xc[:, k], wc)
        if xa2 is not None:
            row['mean_shift_noise'] = (mu['A'] - wmean(xa2, wa2)[k]) / sigma_a[k]
            row['ks_noise'] = weighted_ks(xa[:, k], wa, xa2[:, k], wa2)
        else:
            row['mean_shift_noise'] = np.nan
            row['ks_noise'] = np.nan
        out.append(row)
    return out


records = []
for (fit, suite), got in LOADED.items():
    ab, c, a2 = got['ab'], got['c'], got['a2']
    wa = uniform_weights(len(ab['eta_samples']))
    wb = ab['weights']
    wc = uniform_weights(len(c['eta_samples']))
    wa2 = uniform_weights(len(a2['eta_samples'])) if a2 is not None else None

    eta_names = [f'eta{k + 1}' for k in range(ab['eta_samples'].shape[1])]
    rows = compare('eta', eta_names, ab['eta_samples'], wa, ab['eta_samples'], wb,
                   c['eta_samples'], wc,
                   a2['eta_samples'] if a2 is not None else None, wa2)
    rows += compare('physical', list(ab['names']), ab['samples'], wa, ab['samples'], wb,
                    c['samples'], wc,
                    a2['samples'] if a2 is not None else None, wa2)
    for row in rows:
        row.update(fit=fit, suite=SUITE_LABEL[suite], ess_frac=ab['ess'] / len(ab['eta_samples']))
    records += rows

METRICS = pd.DataFrame(records)[
    ['fit', 'suite', 'space', 'parameter', 'sigma_A', 'mean_shift_A_C', 'z_A_C',
     'mean_shift_B_C', 'z_B_C', 'se_B_C', 'width_ratio_B_C', 'ks_A_C', 'ks_B_C', 'ess_frac']
]
display(METRICS.round(4))


## Section 4: verdicts

Per (fit, suite), taking the worst parameter in $\eta$ space:

* **effect** $=\max_k|{\rm A}-{\rm C}|$ in units of $\sigma_{\rm A}$, with $z_{\rm A-C}$ its
  size in units of its own MCMC uncertainty;
* **closure** $=\max_k|{\rm B}-{\rm C}|$, with $z_{\rm B-C}$ likewise.

The uncertainties are batch-means standard errors of the two chains added in quadrature, so
they measure exactly what is needed: how far two independent chains' posterior means may sit
apart by sampling alone. That makes the seed-2 run unnecessary, and it replaces the fixed
threshold the first version of this notebook used -- with three parameters and a per-parameter
noise of order $0.03\sigma$, a worst-of-three difference near $0.05\sigma$ is *expected* from
sampling, and a fixed $0.05$ cut would report it as a failure.

A fit **passes** when $|z_{\rm B-C}|<3$: B and C agree within MCMC noise. It is
**informative** when $|z_{\rm A-C}|>3$, i.e. the factorization error was measurable at all.
Where it is not, the comparison has no power and says nothing either way.


In [ ]:
Z_PASS = 3.0

summary_rows = []
for (fit, suite), got in LOADED.items():
    sub = METRICS[(METRICS.fit == fit) & (METRICS.suite == SUITE_LABEL[suite])
                  & (METRICS.space == 'eta')]
    iA, iB = sub.z_A_C.abs().idxmax(), sub.z_B_C.abs().idxmax()
    summary_rows.append(dict(
        fit=fit, suite=SUITE_LABEL[suite],
        effect_sigma=abs(sub.mean_shift_A_C[iA]), z_A_C=sub.z_A_C[iA],
        closure_sigma=abs(sub.mean_shift_B_C[iB]), z_B_C=sub.z_B_C[iB],
        noise_sigma=float(sub.se_B_C.max()),
        ks_A_C=float(sub.ks_A_C.max()), ks_B_C=float(sub.ks_B_C.max()),
        ess_frac=float(sub.ess_frac.iloc[0]),
        passes=abs(sub.z_B_C[iB]) < Z_PASS,
        informative=abs(sub.z_A_C[iA]) > Z_PASS,
    ))

SUMMARY = pd.DataFrame(summary_rows).sort_values(['suite', 'fit'])
display(SUMMARY.round(4))

SUMMARY.to_csv(TABLE_DIR / 'cross_vs_reweighting_summary.csv', index=False)
METRICS.to_csv(TABLE_DIR / 'cross_vs_reweighting_metrics.csv', index=False)
print('wrote', TABLE_DIR / 'cross_vs_reweighting_summary.csv')

failed = SUMMARY[~SUMMARY.passes]
if len(failed):
    print('\nFAILING closure (B and C differ by more than MCMC noise):')
    display(failed.round(4))
else:
    print('\nAll available (fit, suite) combinations close within MCMC noise.')

informative = SUMMARY[SUMMARY.informative]
if len(informative):
    print('\nWhere the factorization error was measurable, and what reweighting did to it:')
    for _, r in informative.iterrows():
        print(f"    {r.fit} / {r.suite}: A-C = {r.effect_sigma:.3f} sigma (z = {r.z_A_C:+.1f})"
              f"  ->  B-C = {r.closure_sigma:.3f} sigma (z = {r.z_B_C:+.1f}),"
              f" {100 * (1 - r.closure_sigma / r.effect_sigma):.0f}% removed")
else:
    print('\nNo fit shows a factorization error above MCMC noise; the test has no power here.')


## Section 5: marginals

One panel per parameter. A and C are histograms of their own chains; B is A reweighted. If the
cross branches and the reweighting agree, the B and C curves should lie on top of each other
while A sits visibly off wherever the effect is real.


In [ ]:
def marginal_panels(fit, suite):
    got = LOADED[(fit, suite)]
    ab, c = got['ab'], got['c']
    n_par = ab['eta_samples'].shape[1]
    wa = uniform_weights(len(ab['eta_samples']))
    wc = uniform_weights(len(c['eta_samples']))

    fig, axes = plt.subplots(1, n_par, figsize=(4.2 * n_par, 3.4), squeeze=False)
    for k in range(n_par):
        ax = axes[0][k]
        lo = min(ab['eta_samples'][:, k].min(), c['eta_samples'][:, k].min())
        hi = max(ab['eta_samples'][:, k].max(), c['eta_samples'][:, k].max())
        bins = np.linspace(lo, hi, 60)
        for label, x, w, style in (
            ('A  product', ab['eta_samples'][:, k], wa, dict(color='#888888', ls='-')),
            ('B  reweighted', ab['eta_samples'][:, k], ab['weights'], dict(color='#1F77B4', ls='-')),
            ('C  cross', c['eta_samples'][:, k], wc, dict(color='#D62728', ls='--')),
        ):
            ax.hist(x, bins=bins, weights=w, density=True, histtype='step', lw=1.8,
                    label=label, **style)
        ax.set_xlabel(rf'$\eta_{{{k + 1}}}$')
        ax.set_ylabel('posterior density' if k == 0 else '')
        if k == 0:
            ax.legend(fontsize=9)
    fig.suptitle(f'{fit} — {SUITE_LABEL[suite]}   ({prior_label(SPEC_BY_KEY[fit])})', y=1.02)
    fig.tight_layout()
    return fig


for (fit, suite) in LOADED:
    fig = marginal_panels(fit, suite)
    fig.savefig(FIG_DIR / f'marginals_{fit}_{suite}.pdf', bbox_inches='tight')
    plt.show()


## Section 6: the comparison at a glance

Closure against effect size, with the pass bound as a diagonal band. Points below the band close;
points to the right of it are the ones where the factorization error actually mattered. The
interesting quadrant is bottom-right: a real effect, correctly removed by both routes.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.0))
markers = {'NuWro fake data': 'o', 'Open data': 's'}
for _, row in SUMMARY.iterrows():
    ax.scatter(row.effect_sigma, row.closure_sigma, s=70, marker=markers[row.suite],
               facecolor='none' if not row.passes else '#1F77B4',
               edgecolor='#D62728' if not row.passes else '#1F77B4', lw=1.6,
               label=None, zorder=3)
    ax.annotate(row.fit.replace('minerva_', 'mnv_'), (row.effect_sigma, row.closure_sigma),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

# the pass band is 3 x the MCMC noise on the difference, per fit; draw the largest
bound = float((3 * SUMMARY.noise_sigma).max())
ax.axhspan(0, bound, color='#1F77B4', alpha=0.10, zorder=0)
ax.axhline(bound, color='#1F77B4', lw=1.0, ls=':')
ax.axvline(bound, color='#888888', lw=1.0, ls=':')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'effect size  $\max_k|{\rm A}-{\rm C}|\ [\sigma_{\rm A}]$')
ax.set_ylabel(r'closure  $\max_k|{\rm B}-{\rm C}|\ [\sigma_{\rm A}]$')
ax.set_title('Cross branches vs. chain reweighting')
handles = [plt.Line2D([], [], ls='', marker=m, color='#1F77B4', label=s)
           for s, m in markers.items()]
handles.append(plt.Line2D([], [], ls=':', color='#1F77B4', label='3 x MCMC noise'))
ax.legend(handles=handles, fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'closure_vs_effect.pdf', bbox_inches='tight')
plt.show()
